# Data preparation

This file contains all the following steps:

**constants → read the CSV → build the cohort (traps 2 & 3) → engineer features
→ split (trap 1)**

The analysis notebooks load this with `%run data_prep.ipynb`, which brings all
the constants and functions below into their namespace.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit

#Paths
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_CSV = RAW_DIR / 'diabetic_data.csv'
RAW_DIR.mkdir(parents=True, exist_ok=True)

#Reproducibility
SEED = 42
TEST_SIZE = 0.20
UCI_DATASET_ID = 296  #'Diabetes 130-US Hospitals for Years 1999-2008'

#keys: Never split on encounter_id; group on patient_nbr (making sure the same patient is not in training and test sets)
ENCOUNTER_ID = 'encounter_id'
PATIENT_ID = 'patient_nbr'
TARGET_RAW = 'readmitted'   # '<30', '>30', 'NO'
TARGET = 'readmit_30d'      # binary target I create
POSITIVE_LABEL = '<30'      # Issue 3: HRRP is a 30-day program

#Issue 2: expired/hospice discharge codes (patients who cannot be readmitted)
EXPIRED_HOSPICE_DISPOSITION_IDS = (11, 13, 14, 19, 20, 21)

#Data quality handling
DROP_COLUMNS = ['weight']                       # ~97% missing -> drop, don't impute
MISSING_TOKENS = ['?', 'Unknown/Invalid', 'None', 'NULL', '']
SENSITIVE_FEATURES = ['race', 'gender', 'age', 'payer_code']  #for the fairness audit downstream

#Feature groups
NUMERIC_FEATURES = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
ID_CATEGORICALS = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
DEMOGRAPHIC_CATEGORICALS = ['race', 'gender', 'age', 'payer_code', 'medical_specialty']
LAB_CATEGORICALS = ['max_glu_serum', 'A1Cresult']
MEDICATION_COLUMNS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone']
MED_FLAGS = ['change', 'diabetesMed']
DIAG_COLUMNS = ['diag_1', 'diag_2', 'diag_3']
DIAG_GROUP_COLUMNS = ['diag_1_group', 'diag_2_group', 'diag_3_group']

## 1. Read functions: download the UCI dataset

In [3]:
def download_raw(force=False):
    'Fetch the dataset from UCI and cache it as a CSV (patient_nbr included).'
    if RAW_CSV.exists() and not force:
        print('[data] Using cached raw CSV at', RAW_CSV)
        return pd.read_csv(RAW_CSV, low_memory=False)
    print('[data] Downloading UCI dataset id=', UCI_DATASET_ID)
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=UCI_DATASET_ID)
    frames = [getattr(repo.data, p).reset_index(drop=True)
              for p in ('ids', 'features', 'targets') if getattr(repo.data, p, None) is not None]
    df = pd.concat(frames, axis=1)
    df = df.loc[:, ~df.columns.duplicated()]
    df.to_csv(RAW_CSV, index=False)
    print(f'[data] Cached {len(df):,} rows x {df.shape[1]} cols -> {RAW_CSV}')
    return df

def load_raw():
    'Read the cached raw CSV, downloading it first if it is not there yet.'
    if not RAW_CSV.exists():
        return download_raw()
    return pd.read_csv(RAW_CSV, low_memory=False)

## 2. Cohort: binarize the label (issue 3) and exclude expired/hospice (issue 2)

The naive cohort keeps everyone; the corrected cohort drops expired/hospice
discharges, which are guaranteed negatives. The label is also a 0 (not readmmited or readmitted beyond 30 days) or 1 (readmitted below 30 days).

In [4]:
def binarize_target(df):
    'Issue 3: add binary readmit_30d -- 1 if <30 else 0.'
    out = df.copy()
    out[TARGET] = (out[TARGET_RAW] == POSITIVE_LABEL).astype(int)
    return out

def build_cohort(df, exclude_expired_hospice):
    'Return the modelling cohort and a countable report of what was excluded.'
    out = binarize_target(df)
    n_input = len(out)
    n_excluded = 0
    if exclude_expired_hospice:  #Issue 2
        mask = out['discharge_disposition_id'].isin(EXPIRED_HOSPICE_DISPOSITION_IDS)
        n_excluded = int(mask.sum())
        out = out.loc[~mask].reset_index(drop=True)
    report = {
        'n_input': n_input,
        'n_excluded_expired_hospice': n_excluded,
        'n_output': len(out),
        'base_rate_readmit_30d': round(float(out[TARGET].mean()), 5),
        'excluded_disposition_ids': list(EXPIRED_HOSPICE_DISPOSITION_IDS) if exclude_expired_hospice else [],
    }
    return out, report

## 3. Features: drop junk, group ICD-9 codes, treat missingness as a category

Drop `weight`; group 700+ ICD-9 codes into the 9 clinical buckets from Strack
et al. (2014); treat missingness as its own category; drop zero-variance columns.

In [6]:
def map_icd9_group(code):
    'Map a raw ICD-9 code to one of nine clinical categories (Strack et al. 2014).'
    if code is None or (isinstance(code, float) and np.isnan(code)):
        return 'Missing'
    s = str(code).strip()
    if s == '' or s.lower() == 'nan' or s == '?':
        return 'Missing'
    if s[0] in ('V', 'v', 'E', 'e'):
        return 'Other'
    try:
        num = float(s)
    except ValueError:
        return 'Other'
    if 250 <= num < 251:
        return 'Diabetes'
    icode = int(num)
    if icode in range(390, 460) or icode == 785: return 'Circulatory'
    if icode in range(460, 520) or icode == 786: return 'Respiratory'
    if icode in range(520, 580) or icode == 787: return 'Digestive'
    if icode in range(800, 1000): return 'Injury'
    if icode in range(710, 740): return 'Musculoskeletal'
    if icode in range(580, 630) or icode == 788: return 'Genitourinary'
    if icode in range(140, 240): return 'Neoplasms'
    return 'Other'

def build_features(df):
    'Turn a cohort frame into a model-ready feature matrix X (+ categorical names + meta report).'
    out = df.copy()
    for c in DROP_COLUMNS:
        if c in out.columns:
            out = out.drop(columns=c)
    for raw_col, grouped in zip(DIAG_COLUMNS, DIAG_GROUP_COLUMNS):
        out[grouped] = out[raw_col].map(map_icd9_group)
    candidate = (ID_CATEGORICALS + DEMOGRAPHIC_CATEGORICALS + LAB_CATEGORICALS
                 + MEDICATION_COLUMNS + MED_FLAGS + DIAG_GROUP_COLUMNS)
    categorical_features, dropped_zv = [], []
    for c in candidate:
        if c not in out.columns:
            continue
        (dropped_zv if out[c].nunique(dropna=False) <= 1 else categorical_features).append(c)
    numeric_features = [c for c in NUMERIC_FEATURES if c in out.columns]
    missing_tokens = set(MISSING_TOKENS)
    for c in categorical_features:
        s = out[c].astype('object')
        s = s.where(s.notna(), 'Missing')
        s = s.where(~s.isin(missing_tokens), 'Missing')
        out[c] = s.astype('category')
    X = out[numeric_features + categorical_features].copy()
    meta = {'numeric_features': numeric_features,
            'categorical_features': categorical_features,
            'dropped_zero_variance': dropped_zv}
    return X, categorical_features, meta

## 4. Split: naive (leaky) scheme vs corrected grouped scheme (Issue 1)

`patient_overlap` measures Issue 1 directly: > 0 for the leaky split, 0 for grouped.

In [7]:
def stratified_split(y, test_size=TEST_SIZE, seed=SEED):
    'Naive split: stratified on the label at the encounter level (leaky).'
    sp = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    return next(sp.split(np.zeros(len(y)), y))

def grouped_split(y, groups, test_size=TEST_SIZE, seed=SEED):
    'Corrected split: grouped on patient_nbr (no patient in both train and test).'
    sp = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    return next(sp.split(np.zeros(len(y)), y, groups=groups))

def patient_overlap(groups, train_idx, test_idx):
    'Quantify Issue 1: how many patients appear on both sides.'
    groups = pd.Series(groups).reset_index(drop=True)
    train_p = set(groups.iloc[train_idx].unique())
    test_p = set(groups.iloc[test_idx].unique())
    overlap = train_p & test_p
    return {'n_train_patients': len(train_p), 'n_test_patients': len(test_p),
            'n_overlapping_patients': len(overlap),
            'pct_test_patients_leaked': round(100 * len(overlap) / len(test_p), 2) if test_p else 0.0}

## Sanity check

Confirms the data is present (downloads on first run). This also runs when the
analysis notebooks `%run` this file, so they start with the data guaranteed cached.

In [9]:
_df = load_raw()
print('data_prep loaded |', f'{len(_df):,} rows,', _df[PATIENT_ID].nunique(), 'patients,',
      round(len(_df) / _df[PATIENT_ID].nunique(), 2), 'encounters/patient')


print(_df.head())

data_prep loaded | 101,766 rows, 71518 patients, 1.42 encounters/patient
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)    NaN   
1        149190     55629189        Caucasian  Female  [10-20)    NaN   
2         64410     86047875  AfricanAmerican  Female  [20-30)    NaN   
3        500364     82442376        Caucasian    Male  [30-40)    NaN   
4         16680     42519267        Caucasian    Male  [40-50)    NaN   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metfo